## Member Profiling and Feature Engineering

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import re
from IPython.display import display

### Load Data

In [ ]:
# Load the cleaned dataset
members_df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/cleaned_members.csv')
attendance_df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/event_attendance.csv')

print("Members head:")
display(members_df.head())
print("\nAttendance head:")
display(attendance_df.head())
print(f"\nUnique members in members_df: {members_df['email'].nunique()}")
print(f"Unique members in attendance_df: {attendance_df['email'].nunique()}")

In [ ]:
# Check distinct sectors and job titles
print("Unique sectors count:", members_df['sector'].nunique())
print("Top 15 sectors:")
display(members_df['sector'].value_counts().head(15))

print("\nSample job titles:")
print(members_df['job_title'].dropna().sample(20, random_state=24).tolist())

In [ ]:
# Seniority Extraction Function
def map_seniority(title):
    if pd.isna(title):
        return 'Unknown', 2  # Default mid-tier / neutral
    t = str(title).lower()

    # Executive / Founder / C-level (Level 4)
    if any(k in t for k in ['chief', 'cto', 'ceo', 'cfo', 'cso', 'ciso', 'founder', 'director', 'partner', 'vp', 'vice president', 'head of', 'owner', 'proprietor', 'managing director', 'md']):
        return 'Executive/Leadership', 4
    # Senior / Lead / Principal (Level 3)
    elif any(k in t for k in ['senior', 'lead', 'principal', 'manager', 'architect', 'lecturer', 'advisor', 'consultant']):
        return 'Senior/Management', 3
    # Mid / Professional (Level 2)
    elif any(k in t for k in ['engineer', 'analyst', 'specialist', 'officer', 'developer', 'associate', 'administrator', 'teacher']):
        return 'Mid/Professional', 2
    # Entry / Junior / Student / Academic (Level 1)
    elif any(k in t for k in ['student', 'intern', 'graduate', 'junior', 'apprentice', 'trainee', 'phd', 'scholar']):
        return 'Entry/Student', 1
    else:
        return 'Mid/Professional', 2


members_df['seniority_level_name'], members_df['seniority_score'] = zip(
    *members_df['job_title'].apply(map_seniority))
display(members_df['seniority_level_name'].value_counts())

### Behavioral Aggregation (Event Count & Event Grouping)

In [ ]:
# Clean event names and group by email
# Group events per email into a single string for TF-IDF
event_grouped = attendance_df.groupby('email')['event_name'].apply(
    lambda x: ' '.join(x)).reset_index()
event_counts = attendance_df.groupby('email')['event_name'].nunique(
).reset_index().rename(columns={'event_name': 'event_attendance_count'})

print(f"Number of attendees with event text: {len(event_grouped)}")

In [ ]:
# Merge attendance metrics into members
profile_df = members_df.merge(event_counts, on='email', how='left')
profile_df['event_attendance_count'] = profile_df['event_attendance_count'].fillna(
    0)

# Merge event text
profile_df = profile_df.merge(event_grouped, on='email', how='left')
profile_df['event_name'] = profile_df['event_name'].fillna('none')

# Sector Standardization
top_sectors = members_df['sector'].value_counts().head(8).index.tolist()
profile_df['clean_sector'] = profile_df['sector'].apply(
    lambda s: s if s in top_sectors else ('Other' if pd.notna(s) else 'Unknown'))

display(profile_df['clean_sector'].value_counts())

### Feature Encoding and Scaling

In [ ]:
# TF-IDF on aggregated event themes
tfidf = TfidfVectorizer(max_features=25, stop_words='english')
tfidf_matrix = tfidf.fit_transform(profile_df['event_name'])
tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
print("TF-IDF Features:", tfidf_cols)

In [ ]:
# Scale numeric features
scaler = MinMaxScaler()
norm_attendance = scaler.fit_transform(profile_df[['event_attendance_count']])
norm_seniority = scaler.fit_transform(profile_df[['seniority_score']])

In [ ]:
# One-hot encode Sector and Seniority Level Name
ohe_sector = pd.get_dummies(
    profile_df['clean_sector'], prefix='sec', dtype=float)
ohe_seniority = pd.get_dummies(
    profile_df['seniority_level_name'], prefix='sen', dtype=float)

In [ ]:
# Combine into feature matrix
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_cols)
norm_features = pd.DataFrame({
    'norm_attendance': norm_attendance.flatten(),
    'norm_seniority': norm_seniority.flatten()
})

engineered_features = pd.concat([
    profile_df[['email', 'full_name', 'job_title', 'organization',
                'clean_sector', 'seniority_level_name', 'event_attendance_count']],
    norm_features,
    ohe_sector,
    ohe_seniority,
    tfidf_df
], axis=1)

print("Engineered feature matrix shape:", engineered_features.shape)
display(engineered_features.head())
engineered_features.to_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/engineered_member_profiles.csv', index=False)
print("Saved engineered_member_profiles.csv successfully.")